[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/12_Capacidades_LLM.ipynb)

# Capacidades de un LLM

Hasta ahora usamos LLMs para una cosa a la vez (embeddings, RAG). Aquí vemos, en un solo notebook, **todo lo que un LLM moderno sabe hacer**: no solo contestar texto.

Usamos **Gemini** (`gemini-3.8-flash`), porque es **gratis** — los mismos ejemplos con OpenAI (GPT-5) cuestan dinero, y no todos van a querer pagar para seguir la clase.

Vamos en 9 partes:

1. Llamado simple, roles (`sistema`/`usuario`), uso de tokens
2. Reasoning — qué tanto "piensa" el modelo antes de responder
3. Verbosity — qué tan largo responde (conceptual, Gemini no tiene un botón exacto para esto)
4. Visión — ver una imagen y describirla, o leer texto de ella (OCR)
5. Video — partir un video en imágenes ("frames") y dárselas al modelo
6. Audio — texto a voz, voz a texto, y las dos encadenadas
7. Generación de imágenes — Gemini no lo hace gratis; usamos otro servicio gratis
8. Tools — que el modelo llame funciones tuyas, o busque en Google
9. Structured Outputs — que te regrese JSON con una estructura exacta

**Nota:** esta es una primera versión del notebook; la iremos afinando.

## Archivos que necesitas

1. `calorias.jpeg`
2. `supermarket.mp4`
3. `Greece.mp3`

## Setup — instalar librerías

- `google-genai` — cliente de la API de Gemini.
- `opencv-python-headless` — leer el video y sacarle frames.
- `pillow` — abrir y mostrar imágenes.
- `pydantic` — definir el esquema de los Structured Outputs.

In [ ]:
%pip install -q google-genai opencv-python-headless pillow requests pydantic

## Clave API (Google AI Studio, desde Colab)

1. Entra a [Google AI Studio → API keys](https://aistudio.google.com/apikey) y crea una clave.
2. En Colab, abre el panel de **Secrets** (ícono de llave 🔑 en la barra lateral izquierda) y guárdala con el nombre `GEMINI_API_KEY`. Actívala para este notebook.

In [ ]:
import base64
import io
import json
import wave
from typing import List

import cv2
import matplotlib.pyplot as plt
import requests
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import Audio, display
from PIL import Image
from pydantic import BaseModel

plt.rcParams.update({"figure.dpi": 100, "axes.grid": False})

import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
API_KEY = os.environ["GEMINI_API_KEY"].strip()

MODELO = "gemini-3.8-flash"
MODELO_TTS = "gemini-3.8-flash-tts"

cliente = genai.Client(api_key=API_KEY)
print(f"Cliente listo. Modelo: {MODELO}")

## 1 — Llamado simple, roles y tokens

Lo mínimo: un texto de entrada, un texto de salida.

In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="Hola! ¿Cómo estás?",
)

print(respuesta.text)

### Prompt de sistema, usuario y asistente

El rol `system_instruction` fija cómo se comporta el modelo durante toda la conversación. Los roles `user`/`model` arman el historial — así el modelo "recuerda" lo que ya se dijo.

In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[
        types.Content(role="user", parts=[types.Part.from_text(text="Hola!")]),
        types.Content(role="model", parts=[types.Part.from_text(
            text="Hola! Soy Iber.ia, tu asistente virtual. ¿Cómo te llamas?"
        )]),
        types.Content(role="user", parts=[types.Part.from_text(
            text="Emilio, ¿tú cómo te llamas?"
        )]),
    ],
    config=types.GenerateContentConfig(
        system_instruction=(
            "Eres un asistente de IA que ayuda a los alumnos de la Ibero a resolver sus dudas. "
            "Siempre preséntate como Iber.ia y pregunta el nombre del alumno."
        ),
    ),
)

print(respuesta.text)

### Cuántos tokens usó

Cada llamada trae el conteo de tokens en `usage_metadata` — útil para entender de qué depende el costo (aunque aquí sea gratis).

In [ ]:
print(f"Tokens de entrada:  {respuesta.usage_metadata.prompt_token_count}")
print(f"Tokens de salida:   {respuesta.usage_metadata.candidates_token_count}")
print(f"Tokens totales:     {respuesta.usage_metadata.total_token_count}")

## 2 — Reasoning: qué tanto "piensa" antes de responder

Los modelos de razonamiento pueden generar pasos intermedios antes de la respuesta final — útil en matemáticas, lógica o planeación. Con `thinking_level` controlas qué tanto: `minimal`, `low`, `medium`, `high`. Más nivel = respuestas mejor razonadas, pero más lentas y con más tokens.

In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="""
    Vas a comprar 3 cuadernos de $45 y 2 plumas de $18.
    En la Tienda A hacen 15% de descuento sobre el subtotal y luego aplican IVA de 16%.
    En la Tienda B hacen 10% de descuento a cada artículo por separado y luego aplican IVA de 16%.
    ¿En cuál tienda pagas menos? Da el total final de cada tienda y concluye con una verificación breve (1 oración).
    """,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    ),
)

print(respuesta.text)

In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="""
    ¿Cuántas pelotas de tenis caben en un salón de clases de 8 m x 6 m x 3 m?
    Indica tres supuestos clave (p. ej., diámetro de la pelota, factor de empaque, espacio ocupado por muebles),
    muestra un cálculo aproximado y reporta un rango razonable (entre la mitad y el doble de tu estimación).
    Termina con la cifra que considerarías más plausible.
    """,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)

print(respuesta.text)

In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="""
    Programa estas tres clases en una mañana con dos salones (Azul y Verde) y tres franjas (8-9, 9-10, 10-11):
    Álgebra (A), Biología (B), Historia (H).
    Profes: Lara (puede A o H; no disponible 9-10), Kim (solo B; disponible todo), Soto (solo A; disponible 9-10 o 10-11).
    Reglas: B debe impartirse en el salón Verde; Historia no puede ir 9-10; no puede haber dos clases en el mismo
    salón y franja; cada clase se imparte exactamente una vez.
    Entrega una tabla final (franja, salón, clase, profesor) y 2-3 razones clave que muestren por qué tu asignación
    cumple todas las restricciones. Si fuera imposible, dilo y explica por qué en una frase.
    """,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="high"),
    ),
)

print(respuesta.text)

## 3 — Verbosity: qué tan largo responde

**Verbosity** controla la cantidad de texto que genera el modelo — respuestas largas y detalladas vs. cortas y directas.

Gemini **no tiene un parámetro exacto equivalente**. Lo más parecido:

- **Pedirlo en el prompt:** "responde en una oración" / "sé lo más breve posible".
- **`max_output_tokens`**: pone un tope duro — si el modelo no alcanza a terminar, la respuesta queda cortada a la mitad, no resumida.

No es lo mismo que un botón de "nivel de verbosidad" como en GPT-5, así que aquí lo dejamos solo como concepto, sin código.

## 4 — Visión

### Una imagen de internet

In [ ]:
HEADERS_WIKIMEDIA = {"User-Agent": "IberoDiplomadoNLP/1.0 (uso educativo, sin fines comerciales)"}

resp_img = requests.get(
    "https://commons.wikimedia.org/wiki/Special:FilePath/The_Parthenon_in_Athens.jpg?width=600",
    headers=HEADERS_WIKIMEDIA,
    timeout=20,
)
resp_img.raise_for_status()
imagen_parthenon = Image.open(io.BytesIO(resp_img.content))

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=["Dime qué ves en esta imagen", imagen_parthenon],
)

display(imagen_parthenon)
print(respuesta.text)

### Una imagen propia — OCR

Le pedimos que lea la tabla de información nutricional de `calorias.jpeg` y nos la regrese en formato markdown.

In [ ]:
imagen_nutricion = Image.open("calorias.jpeg")

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=["Extrae la tabla de información nutricional de esta imagen y devuélvela en formato markdown.", imagen_nutricion],
)

display(imagen_nutricion)
print(respuesta.text)

## 5 — Video: partirlo en frames

Gemini puede recibir video directo, pero vamos a hacer **la técnica a mano**: sirve con cualquier modelo que solo entienda imágenes (no todos entienden video nativo). La idea: sacar una foto del video cada N segundos, y mandar esas fotos como si fueran una serie de imágenes.

In [ ]:
def video_a_frames(ruta_video, cada_n_segundos=3):
    """Saca un frame del video cada N segundos y lo regresa como imagen PIL."""
    video = cv2.VideoCapture(ruta_video)
    fps = video.get(cv2.CAP_PROP_FPS)
    intervalo = int(fps * cada_n_segundos)

    frames = []
    contador = 0
    while True:
        ok, frame = video.read()
        if not ok:
            break
        if contador % intervalo == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
        contador += 1
    video.release()
    return frames


frames = video_a_frames("supermarket.mp4", cada_n_segundos=3)
print(f"Frames extraídos: {len(frames)}")

fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for ax, frame in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.axis("off")
for ax in axes.flat[len(frames):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
prompt_video = """
A continuación se te dan fotos tomadas de un video de un supermercado. Extrae los productos con sus precios
en este formato:

Producto:
Precio:
----

Reglas:
- No inventes información.
- No repitas productos si aparecen en más de una foto.
- Si el precio no es visible en una foto, revisa las fotos cercanas; si de plano no aparece, escribe "No visible".
"""

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[prompt_video, *frames],
)

print(respuesta.text)

## 6 — Audio

### Texto a voz (TTS)

**Aviso técnico:** el audio que regresa Gemini es PCM crudo (puros números), no un archivo `.wav` — hay que ponerle el encabezado nosotros con la librería `wave` antes de poder reproducirlo o guardarlo.

In [ ]:
def guardar_wav(pcm_bytes, ruta, canales=1, muestras_por_segundo=24000, ancho_muestra=2):
    with wave.open(ruta, "wb") as f:
        f.setnchannels(canales)
        f.setsampwidth(ancho_muestra)
        f.setframerate(muestras_por_segundo)
        f.writeframes(pcm_bytes)


respuesta = cliente.models.generate_content(
    model=MODELO_TTS,
    contents="Hola a todos los participantes del diplomado de IA de Ibero!",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
            )
        ),
    ),
)

audio_pcm = respuesta.candidates[0].content.parts[0].inline_data.data
guardar_wav(audio_pcm, "salida_tts.wav")
print("Guardado: salida_tts.wav")

Audio("salida_tts.wav")

### Voz a texto (STT)

Le damos `Greece.mp3` directo — sin transcribirlo nosotros primero.

In [ ]:
with open("Greece.mp3", "rb") as f:
    audio_bytes = f.read()

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[
        "¿Qué se dice en esta grabación? Transcríbela y después tradúcela al español.",
        types.Part.from_bytes(data=audio_bytes, mime_type="audio/mp3"),
    ],
)

print(respuesta.text)

### "Habla con Gemini" — las dos encadenadas

GPT-4o puede recibir audio y responder con audio en **una sola llamada**. Con Gemini en este notebook lo hacemos en **dos pasos**: texto → texto, y luego texto → audio. No es exactamente lo mismo (pierdes tono, tiempo real), pero el resultado final — hablarle y que te conteste hablando — es el mismo.

In [ ]:
pregunta = "Hola! ¿Cómo estás? Me llamo Emilio."

respuesta_texto = cliente.models.generate_content(model=MODELO, contents=pregunta)
print("Respuesta en texto:", respuesta_texto.text)

respuesta_audio = cliente.models.generate_content(
    model=MODELO_TTS,
    contents=respuesta_texto.text,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
            )
        ),
    ),
)

guardar_wav(respuesta_audio.candidates[0].content.parts[0].inline_data.data, "respuesta_hablada.wav")
Audio("respuesta_hablada.wav")

## 7 — Generación de imágenes

**Aviso honesto:** Gemini genera imágenes, pero **no es gratis** — lo confirmamos en la página oficial de precios. Para este notebook usamos [Pollinations.ai](https://pollinations.ai/), un servicio gratuito y sin llave — no es el LLM generando la imagen, es otro servicio por completo.

In [ ]:
from urllib.parse import quote

prompt_imagen = "Genera una infografía detallada sobre las redes neuronales artificiales"
url_imagen = f"https://image.pollinations.ai/prompt/{quote(prompt_imagen)}"

resp = requests.get(url_imagen, timeout=60)
resp.raise_for_status()

with open("imagen_generada.png", "wb") as f:
    f.write(resp.content)

Image.open("imagen_generada.png")

## 8 — Tools

### Function calling

Le damos al modelo una función de Python (`obtener_clima`) descrita como "herramienta". El modelo decide **si** y **cómo** llamarla — nosotros la ejecutamos y le devolvemos el resultado.

In [ ]:
def obtener_clima(latitud, longitud):
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": latitud, "longitude": longitud, "current": "temperature_2m"},
        timeout=20,
    )
    return resp.json()["current"]["temperature_2m"]


herramienta_clima = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="obtener_clima",
        description="Obtiene la temperatura actual en grados Celsius para unas coordenadas.",
        parameters={
            "type": "object",
            "properties": {
                "latitud": {"type": "number"},
                "longitud": {"type": "number"},
            },
            "required": ["latitud", "longitud"],
        },
    )
])

pregunta_clima = "¿Cuál es el clima en la CDMX, Tlalpan?"

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=pregunta_clima,
    config=types.GenerateContentConfig(tools=[herramienta_clima]),
)

llamada = respuesta.candidates[0].content.parts[0].function_call
print(f"El modelo quiere llamar a: {llamada.name}({dict(llamada.args)})")

In [ ]:
resultado = obtener_clima(**llamada.args)
print(f"Temperatura real: {resultado}°C")

respuesta_final = cliente.models.generate_content(
    model=MODELO,
    contents=[
        types.Content(role="user", parts=[types.Part.from_text(text=pregunta_clima)]),
        types.Content(role="model", parts=[types.Part.from_function_call(name=llamada.name, args=llamada.args)]),
        types.Content(role="user", parts=[types.Part.from_function_response(
            name=llamada.name, response={"temperatura_celsius": resultado}
        )]),
    ],
    config=types.GenerateContentConfig(tools=[herramienta_clima]),
)

print(respuesta_final.text)

### Tools ya armadas: buscar en Google

No todas las herramientas las escribes tú — Gemini trae **Google Search** integrado, para preguntas que necesitan información de ahorita.

In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="Háblame sobre los mayores eventos económicos de este mes en México",
    config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())]),
)

print(respuesta.text)

## 9 — Structured Outputs

En vez de texto libre, le pedimos al modelo que regrese **JSON con una forma exacta** — útil para meter la respuesta directo a una base de datos o a otro sistema. Definimos esa forma con una clase de `pydantic`.

In [ ]:
class InformacionPersonal(BaseModel):
    nombre_completo: str
    correo: str
    telefono: str
    ubicacion: str


class Experiencia(BaseModel):
    puesto: str
    empresa: str
    fecha_inicio: str
    fecha_fin: str
    responsabilidades: List[str]


class Educacion(BaseModel):
    grado: str
    institucion: str
    anio_fin: int


class CurriculumInfo(BaseModel):
    informacion_personal: InformacionPersonal
    anios_experiencia: int
    experiencia_laboral: List[Experiencia]
    educacion: List[Educacion]
    habilidades_tecnicas: List[str]

In [ ]:
curriculum_texto = """
CURRÍCULUM VITAE

Nombre: Carlos Alberto Martínez López
Correo: carlos.martinez@datamail.com | Teléfono: +52 55 1234 5678
Ubicación: CDMX, México

PERFIL PROFESIONAL
Ingeniero de Datos con 5 años de experiencia en arquitecturas de datos escalables, Big Data y soluciones cloud.

EXPERIENCIA LABORAL

Senior Data Engineer | TechCorp México | Enero 2022 - Presente
- Diseño de pipelines procesando 10TB diarios con Apache Spark y Airflow
- Implementación de data lake en AWS (S3, Glue, Athena)
- Migración a AWS reduciendo costos 40%

Data Engineer | DataSolutions | Marzo 2020 - Diciembre 2021
- Desarrollo de ETL pipelines en Python
- Optimización de SQL mejorando tiempos 60%
- Dashboards en Tableau para stakeholders

EDUCACIÓN
- Maestría en Ciencia de Datos | UNAM | 2021-2023
- Ingeniería en Sistemas | ITESM | 2015-2019

HABILIDADES TÉCNICAS
Python, SQL, Scala, Spark, Kafka, Airflow, AWS, Docker, Kubernetes, Tableau
"""

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=f"Extrae la información de este currículum:\n\n{curriculum_texto}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=CurriculumInfo,
    ),
)

curriculum = respuesta.parsed
print(curriculum.model_dump_json(indent=2))

## Cierre

Un LLM moderno no es solo "un chat". El mismo modelo, con el mismo patrón (`contents` + `config`), sirve para leer imágenes, videos y audio, llamar funciones tuyas, buscar en internet y devolver datos con una forma exacta.

**Lo que no cubrimos gratis:**

- Generación de imágenes con el propio Gemini (usamos otro servicio).
- Audio-a-audio en una sola llamada (lo hicimos en dos pasos).
- Un "nivel de verbosidad" exacto (solo por prompt).

Si en algún punto necesitas algo que Gemini no cubre gratis, recuerda que también existe NVIDIA (`build.nvidia.com`) con varios modelos gratuitos — pero ahí vas a necesitar combinar más de uno (texto + visión por separado), como vimos al evaluar esa opción.